# N10 · The thermal Sunyaev–Zel'dovich effect and Compton-$y$

## How the hot gas in halos imprints itself on the CMB

**Module 3 — Multi-wavelength large-scale-structure cosmology (PhD onboarding).**
The thermal Sunyaev–Zel'dovich (tSZ) effect is one of the four observables combined in the thesis
(galaxies, lensing, X-ray, **SZ**), measured by Planck, SPT and ACT. This notebook builds the
Compton-$y$ observable from the cluster pressure profile and shows its characteristic spectral
signature.

**Key tools:** `astropy.constants`, `astropy.cosmology` (Planck18), `numpy`, `scipy`.


### Learning objectives

1. Define the Compton-$y$ parameter as a line-of-sight integral of the electron pressure.
2. Derive and plot the tSZ spectral distortion $f(x)$ and locate the $\sim$217 GHz null.
3. Implement the generalised-NFW (GNFW) pressure profile of Arnaud et al. (2010).
4. Project the 3-D pressure into a $y(\theta)$ map for a cluster of given mass and redshift.
5. Compute the integrated Compton signal $Y$ and understand why the tSZ is a steep probe of
   $\sigma_8$ (relevant to the thesis joint analysis).


## References

- Sunyaev & Zel'dovich (1972), Comments Astrophys. Space Phys. 4, 173 — original effect
  [ADS:1972CoASP...4..173S](https://ui.adsabs.harvard.edu/abs/1972CoASP...4..173S).
- Carlstrom, Holder & Reese (2002), ARA&A 40, 643 — review
  [arXiv:astro-ph/0208192](https://arxiv.org/abs/astro-ph/0208192).
- Nagai, Kravtsov & Vikhlinin (2007), ApJ 668, 1 — GNFW pressure form
  [arXiv:astro-ph/0703661](https://arxiv.org/abs/astro-ph/0703661).
- Arnaud et al. (2010), A&A 517, A92 — universal pressure profile (REXCESS)
  [arXiv:0910.1234](https://arxiv.org/abs/0910.1234).
- Komatsu & Seljak (2002), MNRAS 336, 1256 — tSZ power spectrum as a probe of $\sigma_8$
  [arXiv:astro-ph/0205468](https://arxiv.org/abs/astro-ph/0205468).
- Planck Collaboration XXII (2016), A&A 594, A22 — all-sky tSZ map
  [arXiv:1502.01596](https://arxiv.org/abs/1502.01596).


## 1. The Compton-$y$ parameter

CMB photons crossing hot ionised gas inverse-Compton scatter off the energetic electrons. The
amplitude of the resulting spectral distortion is the dimensionless **Compton-$y$ parameter**
(Sunyaev & Zel'dovich 1972; Carlstrom, Holder & Reese 2002, eq. 1):

$$\boxed{\;y = \int \sigma_T\, n_e\, \frac{k_B T_e}{m_e c^2}\, dl
= \frac{\sigma_T}{m_e c^2}\int P_e\, dl\;}$$

where $\sigma_T$ is the Thomson cross-section and $P_e = n_e k_B T_e$ is the **electron pressure**.
The temperature change observed at dimensionless frequency $x \equiv h\nu/(k_B T_{\rm CMB})$ is

$$\frac{\Delta T_{\rm SZ}}{T_{\rm CMB}} = f(x)\, y,
\qquad f(x) = x\,\frac{e^x+1}{e^x-1} - 4 = x\coth\!\frac{x}{2} - 4$$

(non-relativistic limit; Sunyaev & Zel'dovich 1972). $f(x)<0$ below the null and $>0$ above it: the
tSZ is a **decrement** in the Rayleigh–Jeans part of the CMB and an increment in the Wien part.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.constants import sigma_T, m_e, c, k_B, G, m_p
from astropy.cosmology import Planck18 as cosmo

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11, 'axes.grid': True,
                     'grid.alpha': 0.3})

T_cmb = cosmo.Tcmb0           # 2.7255 K
h_planck = 6.62607015e-34     # J s (NIST CODATA 2022, exact)

nu = np.linspace(30e9, 600e9, 500)       # Hz
x = (h_planck * nu) / (k_B.value * T_cmb.value)
f_x = x * np.cosh(x/2) / np.sinh(x/2) - 4.0      # = x coth(x/2) - 4

nu_null = nu[np.argmin(np.abs(f_x))] / 1e9
plt.figure()
plt.plot(nu/1e9, f_x)
plt.axhline(0, color='k', lw=0.8)
plt.axvline(nu_null, color='tomato', ls='--', label=f'null at {nu_null:.0f} GHz')
plt.xlabel('frequency [GHz]'); plt.ylabel(r'$f(x)=x\coth(x/2)-4$')
plt.title('tSZ spectral distortion (non-relativistic)'); plt.legend()
plt.savefig('tsz_spectral_function.png', dpi=130); plt.show()
print(f"tSZ null near {nu_null:.0f} GHz (textbook value 217 GHz)")


## 2. The cluster pressure profile (Arnaud et al. 2010)

The thermal pressure of the intracluster medium follows a near-universal **generalised NFW**
(Nagai, Kravtsov & Vikhlinin 2007) shape, calibrated on the REXCESS sample by Arnaud et al. (2010):

$$P(r) = P_{500}\; \mathbb{p}(x), \qquad x = r/R_{500},$$
$$\mathbb{p}(x) = \frac{P_0}{(c_{500}x)^{\gamma}\,\big[1+(c_{500}x)^{\alpha}\big]^{(\beta-\gamma)/\alpha}},$$

with best-fit parameters $[P_0,c_{500},\gamma,\alpha,\beta] = [8.403\,h_{70}^{-3/2},\,1.177,\,0.3081,\,1.0510,\,5.4905]$
(Arnaud et al. 2010, eq. 12). The characteristic pressure is (their eq. 5)

$$P_{500} = 1.65\times10^{-3}\,E(z)^{8/3}
\left(\frac{M_{500}}{3\times10^{14}\,h_{70}^{-1}M_\odot}\right)^{2/3} h_{70}^{2}\;\;{\rm keV\,cm^{-3}}.$$

The tSZ needs the **electron** pressure $P_e = (\mu/\mu_e)\,P$, with mean molecular weights
$\mu\simeq0.59$ and $\mu_e\simeq1.14$ for a fully ionised primordial-abundance plasma, i.e.
$P_e\simeq0.518\,P$ (Arnaud et al. 2010, §2).


In [ ]:
h70 = cosmo.H0.value / 70.0
P0, c500, gamma, alpha, beta = 8.403 * h70**-1.5, 1.177, 0.3081, 1.0510, 5.4905
mu, mu_e = 0.59, 1.14

def R500_of_M(M500, z):
    # spherical overdensity radius: M500 = (4/3) pi 500 rho_c(z) R500^3
    rho_c = cosmo.critical_density(z)
    R3 = (3 * M500 / (4 * np.pi * 500 * rho_c)).to(u.Mpc**3)
    return (R3 ** (1/3)).to(u.Mpc)

def P500_of_M(M500, z):
    E = cosmo.efunc(z)
    M14 = (M500 / (3e14 * h70**-1 * u.Msun)).to_value(u.dimensionless_unscaled)
    return 1.65e-3 * E**(8/3) * M14**(2/3) * h70**2 * u.Unit('keV cm-3')

def P_thermal(r, M500, z):
    x = (r / R500_of_M(M500, z)).to_value(u.dimensionless_unscaled)
    p = P0 / ((c500 * x)**gamma * (1 + (c500 * x)**alpha)**((beta - gamma)/alpha))
    return P500_of_M(M500, z) * p

M500 = 5e14 * u.Msun
z_cl = 0.2
print("R500 =", R500_of_M(M500, z_cl).to(u.Mpc))
print("P500 =", P500_of_M(M500, z_cl))

r = np.logspace(-2, 0.5, 200) * R500_of_M(M500, z_cl)
plt.figure()
plt.loglog((r/R500_of_M(M500, z_cl)).to_value(u.dimensionless_unscaled),
           P_thermal(r, M500, z_cl).to_value(u.Unit('keV cm-3')))
plt.xlabel(r'$r/R_{500}$'); plt.ylabel(r'$P(r)$ [keV cm$^{-3}$]')
plt.title(f'Arnaud GNFW pressure, M500={M500:.0e}, z={z_cl}'); plt.show()


## 3. Projecting to a Compton-$y$ profile

For a projected (sky) radius $b$, integrate the electron pressure along the line of sight $l$:

$$y(b) = \frac{\sigma_T}{m_e c^2}\int_{-\infty}^{\infty} P_e\!\left(\sqrt{b^2+l^2}\right) dl .$$

We truncate the integral at a few $R_{500}$ and convert the projected radius to an angle via the
angular-diameter distance $D_A(z)$.


In [ ]:
from scipy.integrate import quad

R500 = R500_of_M(M500, z_cl)
pref = (sigma_T / (m_e * c**2)).to(u.cm**2 / u.keV)   # sigma_T / m_e c^2  [cm^2/keV]
Mpc_in_cm = (1 * u.Mpc).to_value(u.cm)

def y_of_b(b_Mpc):
    # y(b) = (sigma_T/m_e c^2) * integral of electron pressure along the line of sight.
    lmax = (5 * R500).to_value(u.Mpc)
    def integrand_per_cm(l_Mpc):
        r = np.sqrt(b_Mpc**2 + l_Mpc**2) * u.Mpc
        Pe = (mu / mu_e) * P_thermal(r, M500, z_cl)      # keV cm^-3
        return (pref * Pe).to_value(u.cm**-1)            # cm^-1
    val, _ = quad(integrand_per_cm, -lmax, lmax, limit=100)  # cm^-1 integrated over Mpc
    return val * Mpc_in_cm                                    # dimensionless

b_grid = np.linspace(0.02, 3.0, 40)            # Mpc
y_grid = np.array([y_of_b(b) for b in b_grid])
theta = (b_grid * u.Mpc / cosmo.angular_diameter_distance(z_cl)).to_value(u.dimensionless_unscaled)
theta_arcmin = np.degrees(theta) * 60

plt.figure()
plt.semilogy(theta_arcmin, y_grid)
plt.xlabel(r'$\theta$ [arcmin]'); plt.ylabel('Compton $y$')
plt.title(f'tSZ profile, M500={M500:.0e} Msun, z={z_cl}')
plt.savefig('tsz_y_profile.png', dpi=130); plt.show()
print("central y ~", y_grid[0])


## 4. Integrated Compton signal and the $\sigma_8$ sensitivity

The integrated Compton parameter $Y = \int y\, d\Omega$ measures the total thermal energy of the gas
and correlates tightly with mass — this $Y$–$M$ relation underpins SZ cluster cosmology
(Arnaud et al. 2010; Planck Coll. XXII 2016). Because the abundance of massive clusters is
exponentially sensitive to the amplitude of fluctuations, the tSZ angular power spectrum scales
steeply with the amplitude, $C_\ell^{yy}\propto \sigma_8^{\,7\text{–}8}\,\Omega_b\,h$
(Komatsu & Seljak 2002) — which is exactly why adding tSZ to galaxies + lensing sharpens the joint
$\Omega_m$–$\sigma_8$ constraint in the thesis (see N12).


In [ ]:
# Crude integrated Y: Y = 2 pi \int y(theta) theta dtheta  [steradian]
theta_rad = np.radians(theta_arcmin / 60)
Y = 2 * np.pi * np.trapezoid(y_grid * theta_rad, theta_rad)
print(f"Y(<{theta_arcmin.max():.1f} arcmin) = {Y:.3e} sr")
print(f"          = {Y * (180/np.pi*60)**2:.3e} arcmin^2")


## 5. The $Y$–$M$ slope as an autodiff derivative

The integrated Compton signal obeys $Y_{\rm sph}\propto P_{500}R_{500}^3$. With the self-similar
scalings $P_{500}\propto M^{2/3}E(z)^{8/3}$ and $R_{500}^3\propto M\,E(z)^{-2}$ (Arnaud et al. 2010),
this gives $Y\propto M^{5/3}E(z)^{2/3}$. Writing it as a differentiable JAX function and applying
`jax.grad` returns the slope $5/3$ exactly — the relation underpinning SZ cluster cosmology, here
*derived by differentiation*.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

def logY_sph(lnM, z):
    M = jnp.exp(lnM)
    E = float(cosmo.efunc(z))            # background expansion at z (constant)
    P500 = M**(2/3) * E**(8/3)           # characteristic pressure (arb. units)
    R500_cubed = M * E**(-2)             # ~ R500^3
    return jnp.log(P500 * R500_cubed)    # Y_sph ~ P500 * R500^3 * (fixed GNFW integral)

slope = float(jax.grad(logY_sph, argnums=0)(jnp.log(5e14), 0.2))
print(f"autodiff  d ln Y / d ln M = {slope:.4f}   (self-similar 5/3 = {5/3:.4f})")


## Exercises

1. **Relativistic correction.** For hot clusters ($k_BT_e\gtrsim10$ keV) the spectral function gains an
   $\mathcal{O}(\theta_e=k_BT_e/m_ec^2)$ correction (Itoh et al. 1998). Estimate $\theta_e$ for a 10 keV
   cluster and the fractional change of $f(x)$ at 150 GHz.
2. **Mass scaling.** Recompute the central $y$ for $M_{500}\in\{1,3,5,10\}\times10^{14}M_\odot$. Verify
   the steep dependence on mass and fit a power law $y_0\propto M_{500}^{\,p}$.
3. **Redshift dependence.** The tSZ surface brightness is famously **redshift-independent** (the $y$
   parameter has no $(1+z)^{-4}$ dimming). Compute $y_0$ for the same mass at $z=0.1,0.5,1.0$ and
   discuss what changes (and what does not).
4. **Y–M relation.** Compute $Y_{500}$ (integrated within $R_{500}$) for the mass grid of exercise 2
   and check $Y_{500}\propto E(z)^{2/3}M_{500}^{5/3}$ (self-similar; Arnaud et al. 2010).


## Summary

- Compton-$y = (\sigma_T/m_ec^2)\int P_e\,dl$ measures the integrated electron pressure.
- The spectral signature $f(x)=x\coth(x/2)-4$ is a decrement below and increment above the $\sim$217 GHz null.
- The Arnaud (2010) GNFW pressure profile turns a halo mass into a $y(\theta)$ map.
- $Y$ tracks the gas thermal energy; the tSZ power spectrum is a steep probe of $\sigma_8$.
- Autodiff of the self-similar model returns the $Y$–$M$ slope $5/3$ exactly.

**Next (N11):** the *same* hot gas seen in X-rays, the fourth observable of the thesis.
